# 07_requests.ipynb
1. `%uv add -q requests` -> 앞에 %가 붙어있으면 python 셀 이지만, 터미널에 실행할 명령어
2. `pip install requests` -> 우리는 최신기술 uv쓸거라 상황에 따라 이거 쓸 수 도 있음

In [1]:
import requests

URL = 'https://www.dhlottery.co.kr/lt645/selectPstLt645Info.do'

res = requests.get(URL)
ram = "성훈"

# 데이터 덩어리를 해석 할 수 있는(ex dict 형태로 바뀌는 등) 형태로 바꾸는 작업이 파싱(parsing) 이라고 불림
type(res.text)   #str -> 단순 글자 상태 -> 파싱 안된 데이터
type(res.json()) #dict -> key,value 값으로 받을 수 있어서(표 형식) 원하는 데이터만 뽑아 낼 수 있다 -> 파싱 된 데이터

raw_data = res.text
parsing_data = res.json()

res.json()

# 1인당 1등 당첨 금액 = 1197258718, key값 : #rnk1WnAmt

print(parsing_data['data']['list'][0]['rnk1WnAmt'])
res.json()

1197258718


{'resultCode': None,
 'resultMessage': None,
 'data': {'list': [{'winType0': 0,
    'winType1': 10,
    'winType2': 12,
    'winType3': 1,
    'gmSqNo': 5133,
    'ltEpsd': 1238,
    'tm1WnNo': 2,
    'tm2WnNo': 13,
    'tm3WnNo': 18,
    'tm4WnNo': 32,
    'tm5WnNo': 38,
    'tm6WnNo': 42,
    'bnsWnNo': 22,
    'ltRflYmd': '20260822',
    'rnk1WnNope': 23,
    'rnk1WnAmt': 1197258718,
    'rnk1SumWnAmt': 27536950514,
    'rnk2WnNope': 84,
    'rnk2WnAmt': 54636807,
    'rnk2SumWnAmt': 4589491788,
    'rnk3WnNope': 3235,
    'rnk3WnAmt': 1418700,
    'rnk3SumWnAmt': 4589494500,
    'rnk4WnNope': 156141,
    'rnk4WnAmt': 50000,
    'rnk4SumWnAmt': 7807050000,
    'rnk5WnNope': 2549183,
    'rnk5WnAmt': 5000,
    'rnk5SumWnAmt': 12745915000,
    'sumWnNope': 2708666,
    'rlvtEpsdSumNtslAmt': 57268901802,
    'wholEpsdSumNtslAmt': 114537798000,
    'excelRnk': '1등'}]}}

In [2]:
lt_data = parsing_data['data']['list'][0]
lucky = []
bonus = 0

#딕셔너리 순회도 가능하다
for k,v in lt_data.items():
  if 'tm' in k:
   lucky.append(v)
  elif 'bns' in k:
    bonus = v

print(lucky)
print(bonus)

[2, 13, 18, 32, 38, 42]
22


In [3]:
# Main Mission 
# 랜덤하게 뽑은 번호 6개와, 실제 당첨번호를 비교하여 몇등인지 출력하는 프로그램
# (추가미션) 함수로 만들기

# 로또 RULE) 
# 1등 숫자 6개 같음
# 2등: 숫자 5개 같고 + 나머지 하나가 보너스 번호
# 3등 ~ 5등 : 숫자 5개, 4개, 3개 같음

import random

# 랜덤 번호 추출 함수
def random_nums():
 return random.sample(range(1,46), 6)

# URL request로 당첨 로또 번호 불러오는 함수  
def print_lt_nums():
 lucky_nums = []#보너스 번호 포함된 로또 번호들

 for k,v in lt_data.items():
   if 'tm' in k:
    lucky_nums.append(v)
   elif 'bns' in k:
     lucky_nums.append(v)

 return lucky_nums

# 보너스 번호 불러 오는 함수
def print_bns_num():
 return lt_data['bnsWnNo']

# 당첨 등수 출력 함수
def check_win(random_nums, lucky_nums, bns_num):
 check_num = 0
 is_bns = False
 #로또 번호 대조
 for random_num in random_nums:
  if random_num in lucky_nums:
   check_num += 1
  if random_num == bns_num:
   is_bns = True
 #등수 출력
 if check_num == 6 and is_bns == False:
  return '1등'
 elif check_num == 6 and is_bns == True:
  return '2등'
 elif check_num >= 5:
  return '3등'
 elif check_num >= 4:
  return '4등'
 elif check_num >= 3:
  return '5등'
 else:
  return '꽝'

print(check_win(random_nums(), print_lt_nums(), print_bns_num()))
print(check_win({2,13,18,32,22}, print_lt_nums(), print_bns_num()))



꽝
3등


## API 키 관리
1. `uv add python-dotnev`
2. 모든 키 파일은 `.env` 파일에 보관
3. 소스코드에서는 `load_dotenv()와` `os.getenv()`를 사용하여 불러옴

In [4]:
import os
from dotenv import load_dotenv
# .env 파일 불러오기
load_dotenv()
# 불러온 파일에서 원하는 key 꺼내기
NAVER_CLIENT_ID = os.getenv('NAVER_CLIENT_ID')
NAVER_CLIENT_SECRET = os.getenv('NAVER_CLIENT_SECRET')


In [5]:
#네이버 API 식 요청 방식(인증 관련 정보)
BASE_URL = 'https://naverapihub.apigw.ntruss.com'
NEWS_URL = '/search/v1/news'

headers = {
    'X-NCP-APIGW-API-KEY-ID': NAVER_CLIENT_ID,
    'X-NCP-APIGW-API-KEY': NAVER_CLIENT_SECRET
}

In [6]:
import requests
# 클래식한 쿼리 파라미터
URL = BASE_URL + NEWS_URL

# 쿼리 파라미터를 dict 로 작성
params = {
  'query': '엔화',
  'sort': 'sim',
  'dispaly' : 5, #기사 100개 모아서
}

print(URL)

res = requests.get(URL, headers=headers, params=params)
res.json()

https://naverapihub.apigw.ntruss.com/search/v1/news


{'lastBuildDate': 'Thu, 27 Aug 2026 11:04:13 +0900',
 'total': 434722,
 'start': 1,
 'display': 10,
 'items': [{'title': '<b>엔화</b>, 美 장기금리 상승에 1달러=159엔대 전반 하락 출발',
   'originallink': 'https://www.newsis.com/view/NISX20260827_0003764746',
   'link': 'https://n.news.naver.com/mnews/article/003/0014151350?sid=104',
   'description': '<b>엔화</b> 환율은 27일 미국 물가지표가 인플레 우려를 부르면서 장기금리 하락으로 엔 매도, 달러 매수가 유입해 1달러=159엔대 전반으로 내려 시작했다. 도쿄 외환시장에서 <b>엔화</b> 환율은 이날 오전 8시30분 시점에... ',
   'pubDate': 'Thu, 27 Aug 2026 09:55:00 +0900'},
  {'title': '“<b>엔화</b> 저평가됐다”…호주 2위 연기금이 달러 약세·<b>엔화</b>강세에 베팅한...',
   'originallink': 'https://www.mk.co.kr/article/12137536',
   'link': 'https://n.news.naver.com/mnews/article/009/0005726477?sid=101',
   'description': '판단하며 <b>엔화</b> 강세에 베팅하고 나섰다고 블룸버그통신이 26일(현지시간) 보도했다. 약 3700억 호주달러(약 365조원)를 굴리는 ART의 지미 루카 수석 포트폴리오 매니저는 블룸버그와 인터뷰에서 “지난 6개월간 <b>엔화</b>... ',
   'pubDate': 'Thu, 27 Aug 2026 10:44:00 +0900'},
  {'title': '<b>엔화</b> 약세에 日銀 긴축 빨라지나…9월 금리인상론 급부상',
   'origi

In [7]:
# 100개 기사를 모아서
# title에 <b>, </b> 이상한 태그 없애기
# 조건 : link URL이 naver 뉴스인 애들만 모아야 함.(100개가 안 될 수 있습니다).
# 간략히 다음과 같은 모양으로 만들기
# news 변수 내용을 csv 로 export 하기

'''
news = [
  {'title': '제목제목', 'link': 'https://skdflajsdklf.com'},
  {'title': '제목제목', 'link': 'https://skdflajsdklf.com'},
  {'title': '제목제목', 'link': 'https://skdflajsdklf.com'},
  {'title': '제목제목', 'link': 'https://skdflajsdklf.com'},
  {'title': '제목제목', 'link': 'https://skdflajsdklf.com'},
  {'title': '제목제목', 'link': 'https://skdflajsdklf.com'}
]
'''





"\nnews = [\n  {'title': '제목제목', 'link': 'https://skdflajsdklf.com'},\n  {'title': '제목제목', 'link': 'https://skdflajsdklf.com'},\n  {'title': '제목제목', 'link': 'https://skdflajsdklf.com'},\n  {'title': '제목제목', 'link': 'https://skdflajsdklf.com'},\n  {'title': '제목제목', 'link': 'https://skdflajsdklf.com'},\n  {'title': '제목제목', 'link': 'https://skdflajsdklf.com'}\n]\n"

## Parsing
1. JSON 문자열 -> dict 로 해석
2. HTML 문자열 -> 구조화 필요 (`Beautifulsoup4`)

In [8]:
# uv add beautifulsoup4
import requests
from bs4 import BeautifulSoup

URL = 'https://n.news.naver.com/article/008/0005405342'

def extract_naver_news(url):
   # 네이버 뉴스 아니면 에러발생
   if 'n.news.naver.com' not in url:
     #예외처리로 에러 발생 시키는 코드
     raise Exception('네이버 뉴스가 아닙니다')

   res = requests.get(url)

   #res.text 를 해석 완료!
   soup = BeautifulSoup(res.text, 'html.parser')

   #해석한 HTML 에서 '#dic_area' 선택자로 추출 -> 글자만 뽑아서 -> 양옆 공백(엔터, 스페이스) 삭제
   news_text = soup.select_one('#dic_area').text.strip()

   return news_text



extract_naver_news(URL)


"목요일인 27일 전국 곳곳에 비가 내리겠다. 낮 기온이 최대 34도까지 오르는 등 더위는 계속되겠다. /사진=뉴스1 목요일인 오늘(27일) 전국 곳곳에 비나 소나기가 내리겠다. 낮 기온은 최대 34도까지 오르는 등 무더위는 계속되겠다.기상청에 따르면 이날 아침 최저기온은 21~26도, 낮 최고기온은 29~34도로 예보됐다. 최고 체감온도는 33도 안팎까지 오르겠다.아침부터 낮 사이 경기 북부와 강원에는 5~30㎜의 비가 내리겠다. 충청권과 전북, 대구·경북에는 오전부터 저녁 사이 5~50㎜의 소나기가 내리는 곳이 있겠다.전남권과 경남권, 남해안, 제주도에도 밤까지 비가 이어지겠다. 예상 강수량은 광주·전남과 부산·울산·경남 10~60㎜, 제주도 5~30㎜다. 특히 경남 남해안에는 80㎜ 이상의 많은 비가 내리는 곳도 있겠다.비나 소나기가 내리는 지역에서는 일시적으로 기온이 낮아지겠으나 비가 그친 뒤에는 높은 습도와 함께 다시 무더위가 이어지겠다.주요 도시 예상 최저기온은 △서울 25도 △인천 24도 △춘천 23도 △강릉 24도 △대전 24도 △대구 24도 △전주 25도 △광주 26도 △부산 26도 △여수 26도 △제주 27도 △울릉도·독도 25도 등이다.예상 낮 최고기온은 △서울 31도 △인천 30도 △춘천 31도 △강릉 30도 △대전 33도 △대구 33도 △전주 33도 △광주 33도 △부산 32도 △여수 31도 △제주 34도 △울릉도·독도 29도 등이다.미세먼지 농도는 전 권역에서 '좋음'~'보통' 수준을 보이겠다."

In [ ]:
# 1. 특정 주제로 Naver News 연관도 순으로 5개 뽑기
# 2. Naver 뉴스 링크를 통해서 본문만 추출하기
# 3. 최종 결과형식

'''
news = [
  {'title': '제목제목', 'link': '네이버뉴스링크', 'content': '추출한 본문', 'comment':'좋아요'},
  {'title': '제목제목', 'link': '네이버뉴스링크', 'content': '추출한 본문', 'comment':'좋아요'},
  {'title': '제목제목', 'link': '네이버뉴스링크', 'content': '추출한 본문', 'comment':'좋아요'},
  {'title': '제목제목', 'link': '네이버뉴스링크', 'content': '추출한 본문', 'comment':'좋아요'}
]
'''

<Response [200]>
